# Week 2 · Lab: Looking inside a transformer

Companion to [Looking Inside a Transformer](https://revanthreddy-hai.github.io/the-llm-residency/week02.html).
Everything the essay measures, reproduced end to end on a free CPU: the parameter census, the
indirect-object sentence, the attention weights of all 144 heads, and the logit lens at every layer.
First run downloads about 0.5 GB of model weights.

*Self-checks are `assert` cells. If one fails, something real changed — read it before moving on.*


## 1 · Set up the environment


In [ ]:
# Colab and fresh environments install the pinned dependencies on first run.
# Where they are already present, this cell just confirms the imports resolve.
# If versions differ (e.g. stock Colab), pip fetches the pinned wheels: ~0.5-1 GB extra.
from importlib.metadata import version, PackageNotFoundError

DEPENDENCIES = {"torch": "torch==2.13.0", "transformers": "transformers==5.16.1"}

def installed(pip_spec):
    name, pinned = pip_spec.split("==")
    try:
        return version(name) == pinned
    except PackageNotFoundError:
        return False

mismatched = [spec for spec in DEPENDENCIES.values() if not installed(spec)]
if mismatched:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *mismatched])

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)


## 2 · One sentence, two prompts

The test sentence is the indirect-object identification task from Wang et al. (2022): the model
must hand the drink to whichever name was mentioned *once*. If GPT-2 were completing a frequency
pattern, swapping which name repeats would not swap the answer. It does.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GPT2_REV = "607a30d783dfa663caf39e06633721c8d4cfcd7e"  # pin the exact weights the essay measured

tok = AutoTokenizer.from_pretrained("gpt2", revision=GPT2_REV)
model = AutoModelForCausalLM.from_pretrained(
    "gpt2", revision=GPT2_REV, output_attentions=True, output_hidden_states=True,
    attn_implementation="eager")
model.eval()

PROMPTS = {
    "john_gives": "When Mary and John went to the store, John gave a drink to",
    "mary_gives": "When Mary and John went to the store, Mary gave a drink to",
}

def top5(prompt):
    enc = tok(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model(**enc)
    probs = torch.softmax(out.logits[0, -1], dim=-1)
    top = torch.topk(probs, 5)
    return [(tok.decode([i]), round(float(p), 4)) for p, i in zip(top.values, top.indices)], out, enc

for name, prompt in PROMPTS.items():
    guesses, _, _ = top5(prompt)
    print(f"{name}: {guesses}")


In [ ]:
# self-check: the answer tracks which name repeats, not which name is "likelier"
guesses_j, out, enc = top5(PROMPTS["john_gives"])
guesses_m, _, _ = top5(PROMPTS["mary_gives"])
assert guesses_j[0][0] == " Mary" and guesses_j[0][1] > 0.40, guesses_j[0]
assert guesses_m[0][0] == " John" and guesses_m[0][1] > 0.25, guesses_m[0]
p = dict(guesses_j)
pj = p.get(" John", float("nan"))
assert p[" Mary"] / (pj if pj == pj else 1e-9) > 5, "Mary should beat John by a wide margin"
print("ok — Mary at", guesses_j[0][1], "vs John at", pj, "; control flips to John at", guesses_m[0][1])


## 3 · Weigh the machine

One block holds an attention layer and an MLP. Before looking at what they do, count what they cost.


In [ ]:
params = dict(model.named_parameters())
def count(*names):
    return sum(params[n].numel() for n in names)

blk = "transformer.h.0."
attn = count(blk + "attn.c_attn.weight", blk + "attn.c_attn.bias",
             blk + "attn.c_proj.weight", blk + "attn.c_proj.bias")
mlp = count(blk + "mlp.c_fc.weight", blk + "mlp.c_fc.bias",
            blk + "mlp.c_proj.weight", blk + "mlp.c_proj.bias")
ln = count(blk + "ln_1.weight", blk + "ln_1.bias", blk + "ln_2.weight", blk + "ln_2.bias")
wte = count("transformer.wte.weight")
wpe = count("transformer.wpe.weight")
total = sum(q.numel() for q in model.parameters())

print(f"attention / block : {attn:>10,}")
print(f"MLP / block       : {mlp:>10,}  ({mlp / (attn + mlp + ln):.1%} of the block)")
print(f"layernorms / block: {ln:>10,}")
print(f"embeddings        : {wte:>10,}  + positions {wpe:,}")
print(f"total             : {total:>10,}")


In [ ]:
# self-check: the MLP holds two thirds of every block
assert attn == 2_362_368 and mlp == 4_722_432 and ln == 3_072
assert total == 124_439_808
assert abs(mlp / (attn + mlp + ln) - 0.666) < 0.001
print("ok — block =", f"{attn + mlp + ln:,}", "params, MLP share 66.6%")


## 4 · Where does the last position look?

Attention weights from the final position (the “to” that must choose a name), for all 12 heads of
all 12 layers. Three habits to find: previous-token heads, first-token sinks, and the late-layer
heads that carry the answer.


In [ ]:
att = [a[0, :, -1, :] for a in out.attentions]          # 12 layers x [12 heads, seq]
toks = [tok.decode([t]) for t in enc["input_ids"][0]]
i_mary, i_first, i_prev = 1, 0, len(toks) - 2

sink = sum(1 for L in range(12) for H in range(12) if float(att[L][H][i_first]) > 0.5)
prev = sum(1 for L in range(12) for H in range(12) if int(att[L][H].argmax()) == i_prev)
mean_first = float(torch.stack(att).mean(dim=(0, 1))[i_first])

by_mary = sorted(((float(att[L][H][i_mary]), L + 1, H + 1)
                  for L in range(12) for H in range(12)), reverse=True)
print(f"heads spending >half their weight on the first token: {sink} of 144")
print(f"heads whose top target is the previous token: {prev}")
print(f"average weight on the first token: {mean_first:.3f}")
print("heads with the most weight on ' Mary':",
      [(f"L{L}H{H}", round(w, 3)) for w, L, H in by_mary[:3]])


In [ ]:
# self-check: the sink is the norm and the name movers sit late in the stack
assert sink >= 60, sink                       # most heads idle on the first token
assert mean_first > 0.4, mean_first
assert prev >= 10, prev              # previous-token heads are a stable population
top_w, top_L, top_H = by_mary[0]
assert top_w > 0.5 and top_L >= 10, (top_w, top_L, top_H)
print(f"ok — strongest Mary head is L{top_L}H{top_H} at {top_w:.3f}; {sink}/144 heads sink to token 0")


Wang et al. (2022) reverse-engineered this exact task in this exact model. Two of the heads above
match their **name mover heads**, the copiers (layer 10 heads 7 and 10, 1-indexed). The two
*strongest* lookers — layer 11 head 8 and layer 12 head 11 — are their **negative name mover
heads**: they attend to Mary just as hard and write *against* her. Attention weights say where a
head looks, not which way it pushes; only causal interventions of the kind Wang et al. ran can
tell copiers from suppressors.


## 5 · The logit lens

The residual stream is a running draft: after any block, apply the final layernorm and the output
head to the vector as it stands, and read the model's guess-so-far.

One trap, found the hard way: `hidden_states[-1]` is returned **already normalized** in HF GPT-2.
Normalize it a second time and the layer-12 readout stops matching the model's real output — the
check in the self-check cell below is how the bug announced itself.


In [ ]:
id_mary = tok.encode(" Mary")[0]
ln_f, W_U = model.transformer.ln_f, model.transformer.wte.weight

lens = []
with torch.no_grad():
    for L, h in enumerate(out.hidden_states):
        v = h[0, -1] if L == 12 else ln_f(h[0, -1])   # last entry is already post-ln_f
        pr = torch.softmax(v @ W_U.T, dim=-1)
        t1 = int(pr.argmax())
        lens.append((L, tok.decode([t1]), round(float(pr[t1]), 4),
                     int((pr > pr[id_mary]).sum()) + 1))

print(" layer | top guess     |    p  | rank of ' Mary'")
for L, t, p, r in lens:
    print(f"  {L:>4} | {t!r:13} | {p:.3f} | {r:>6,}")


In [ ]:
# self-check 1: the lens at layer 12 must reproduce the model's actual output exactly
real = torch.softmax(out.logits[0, -1], dim=-1)
lens_final = torch.softmax(out.hidden_states[-1][0, -1] @ W_U.T, dim=-1)
assert torch.allclose(real, lens_final, atol=1e-4), "probe broken: layer-12 lens != model output"

# self-check 2: the answer arrives late and in a jump, not a climb
ranks = {L: r for L, _, _, r in lens}
assert ranks[0] > 10_000        # unrecognizable at the embeddings
assert ranks[9] > 10             # still generic after layer 9
assert ranks[10] <= 3            # the jump
assert ranks[11] == 1 and ranks[12] == 1
print("ok — rank trajectory:", [ranks[L] for L in (0, 5, 9, 10, 11, 12)])


## What you just did

- Confirmed GPT-2 resolves *who gets the drink* from context alone, and flips its answer when the
  repeated name flips — Mary at 0.446 against John at 0.060, and the reverse under the control.
- Weighed one block: attention 2,362,368 parameters, MLP 4,722,432 — the lookup that moves
  information between positions is the *smaller* part of every block.
- Censused all 144 heads on one sentence: 66 idle on the first token (the attention sink), a
  previous-token head at 0.9999, and late-layer heads locked onto " Mary" — per Wang et al., a mix
  of name movers that copy her and negative name movers that write against her.
- Read the residual stream mid-flight with the logit lens and watched the answer arrive at layer 11
  of 12 — a cliff, not a climb — then verified the probe against the model's real output.

Next week the encoder side gets its turn: representation models read, generation models write, and
classification wants a reader.
